# Idempotent Pipelines: Deterministic IDs, Atomic Partitions, Run Ledgers

## What you'll practice

An idempotent job can run twice (retries, backfills, a nervous on-call engineer) and produce the same result as running once. Techniques: deterministic identifiers,
overwrite-a-partition instead of append, append-with-dedup, and a run ledger that records what succeeded with which inputs.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/etl/test_08_06_idempotent_pipelines.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/08_data_engineering_python/06_idempotent_pipelines.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Deterministic run IDs

**Implement:** `run_id(job_name: str, params: dict) -> str`

Return `f"{job_name}-{hash12}"` where `hash12` is the first 12 hex chars of SHA-256 over the **canonical** JSON of params (`sort_keys=True`, `separators=(",", ":")`, `default=str`). Same params in any key order → same id.

**Input:** `job_name: str, params: dict`

**Expected output:** `str`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_run_id` in `tests/etl/test_08_06_idempotent_pipelines.py`.

**Check:** `pytest tests/etl/test_08_06_idempotent_pipelines.py -k test_run_id`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Overwrite a partition atomically

**Implement:** `write_partition(root, partition: dict[str, str], rows: list[dict], filename: str = 'part-00000.jsonl') -> Path`

Write rows as JSONL into `root/k=v/.../filename`, **replacing** whatever the partition contained before. Do it safely: write into a sibling temp directory
(`.tmp-<uuid>`), then remove the old partition directory and rename the temp one into place. Re-running with the same rows yields byte-identical files (`sort_keys=True`). Return the partition dir.

**Input:** `root, partition: dict[str, str], rows: list[dict], filename: str = 'part-00000.jsonl'`

**Expected output:** `Path`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_write_partition` in `tests/etl/test_08_06_idempotent_pipelines.py`.

**Check:** `pytest tests/etl/test_08_06_idempotent_pipelines.py -k test_write_partition`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Append with deduplication

**Implement:** `append_unique(path, records: list[dict], key: str) -> int`

Append records to a JSONL file only if their `key` isn't already present in the file **or earlier in this batch**. Create the file if missing. Return how many were appended. Running twice with the same batch appends 0 the second time.

**Input:** `path, records: list[dict], key: str`

**Expected output:** `int`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_append_unique` in `tests/etl/test_08_06_idempotent_pipelines.py`.

**Check:** `pytest tests/etl/test_08_06_idempotent_pipelines.py -k test_append_unique`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · A run ledger

**Implement:** `class RunLedger`

`RunLedger(path, stale_after_seconds=3600, clock=lambda: datetime.now(timezone.utc))` persists (JSON) one entry per `(job, partition)`:
`{"status": "running"|"success"|"failed", "input_hash", "started_at", "finished_at", "stats", "error", "attempts"}`.
- `should_run(job, partition, input_hash, force=False)` → `False` only if the last status is `success` with the **same** input hash (and not forced); also `False` if another run is
  `running` and not stale (started less than `stale_after_seconds` ago); stale `running` entries count as failed.
- `start(job, partition, input_hash)` (increments attempts), `succeed(job, partition, stats)`, `fail(job, partition, error)`; each persists immediately.
- `history()` → dict copy of all entries keyed `"job|partition"`.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_run_ledger_lifecycle, test_run_ledger_failure` in `tests/etl/test_08_06_idempotent_pipelines.py`.

**Check:** `pytest tests/etl/test_08_06_idempotent_pipelines.py -k "test_run_ledger_lifecycle or test_run_ledger_failure"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · An idempotent daily job end-to-end · 🧩 Mini-project

**Implement:** `run_daily_job(daily_dir, output_root, ledger_path, force: bool = False) -> dict`

Process `datasets/daily/dt=*/` partitions so that reruns are free and changes are detected:
1. For each partition directory, compute an **input hash** = SHA-256 over the sorted `(file name, file sha256)` pairs of its `*.csv` files.
2. Ask a `RunLedger` whether to run (`job="daily_orders"`, partition = the `dt` value). Skip if not needed.
3. Otherwise: read all CSVs in the partition, transform rows (reuse your `transform_order` logic; bad rows → rejects), dedupe by `order_id` keeping latest `updated_at`,
   write the partition atomically to `output_root/orders/dt=<date>/part-00000.jsonl` and rejects to `output_root/rejects/dt=<date>/part-00000.jsonl` (even if empty), then mark success with stats.
   On any unexpected exception mark the ledger failed and continue with the next partition.
Return `{"processed": [dates], "skipped": [dates], "failed": [dates], "stats": {date: {"read", "loaded", "rejected", "duplicates"}}}`.
Run twice: the second run must skip everything. Then edit one file in a temp copy and confirm only that partition reruns.

**Input:** `daily_dir, output_root, ledger_path, force: bool = False`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_run_daily_job` in `tests/etl/test_08_06_idempotent_pipelines.py`.

**Check:** `pytest tests/etl/test_08_06_idempotent_pipelines.py -k test_run_daily_job`

**YOUR CODE ↓**